In [1]:
!pip install -q -U transformers accelerate gguf sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 4.0 MB/s eta 0:00:00


In [2]:
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_id = "abhinav0231/Lily-1.5b-v0.3"

tokenizer = AutoTokenizer.from_pretrained(repo_id)

model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("✅ model + tokenizer loaded")
print("device:", model.device)

Lily-1.5b-v0.3-Q4_K_M.gguf:   0%|          | 0.00/986M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Converting and de-quantizing GGUF tensors...:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ model + tokenizer loaded
device: cuda:0


In [3]:
TEST_CASES = [
    {
        "name": "easy_arithmetic",
        "user": "What is 15% of 840? Return output exactly in <think> and <answer> tags."
    },
    {
        "name": "egg_problem",
        "user": (
            "Janet's ducks lay 16 eggs per day. She eats 3 for breakfast and "
            "bakes muffins with 4 eggs. She sells the rest at $2 per egg. "
            "How much does she make per day? Return output exactly in "
            "<think> and <answer> tags."
        )
    },
    {
        "name": "speed_distance_conversion",
        "user": (
            "A train travels 120 km in 1.5 hours. What is its average speed in meters per second? "
            "Show your step-by-step calculation inside <think> tags and final numeric answer inside <answer> tags."
        )
    },
    {
        "name": "classic_bat_ball_trick",
        "user": (
            "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. "
            "How much does the ball cost in dollars? Reason inside <think> and answer inside <answer>."
        )
    },
    {
        "name": "python_algorithm",
        "user": (
            "Write an efficient Python function to find the length of the longest palindromic substring in a string. "
            "Provide explanation in <think> and clean code in <answer>."
        )
    },
    {
        "name": "structured_instruction",
        "user": (
            "List 3 renewable energy sources. Start each bullet point with an emoji and keep each explanation under 15 words. "
            "Format reasoning in <think> and bullet list in <answer>."
        )
    }
]

SYSTEM = (
    "You are a precise, helpful assistant. Always reason step by step "
    "inside <think></think> tags, then write your final answer inside <answer></answer> tags."
)

def extract_stats(text):
    return {
        "has_think_open": "<think>" in text,
        "has_think_close": "</think>" in text,
        "has_answer_open": "<answer>" in text,
        "has_answer_close": "</answer>" in text,
        "think_count": text.count("<think>"),
        "thinking_count": text.count("<thinking>"),
        "answer_count": text.count("<answer>"),
        "im_end_count": text.count("<|im_end|>"),
    }

def render_prompt(user_text):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def run_case(case, max_new_tokens=256, temperature=0.2, do_sample=False):
    prompt = render_prompt(case["user"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    decoded_raw = tokenizer.decode(gen_ids, skip_special_tokens=False)
    decoded_clean = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return {
        "name": case["name"],
        "prompt": prompt,
        "decoded_raw": decoded_raw,
        "decoded_clean": decoded_clean,
        "raw_stats": extract_stats(decoded_raw),
        "clean_stats": extract_stats(decoded_clean),
    }

In [4]:
results = []
for case in TEST_CASES:
    out = run_case(case)
    results.append(out)

    print("=" * 80)
    print("CASE:", out["name"])
    print("-" * 80)
    print("PROMPT PREVIEW:")
    print(out["prompt"][:800])
    print("\nRAW OUTPUT:")
    print(out["decoded_raw"][:2000])
    print("\nCLEAN OUTPUT:")
    print(out["decoded_clean"][:2000])
    print("\nRAW STATS:", out["raw_stats"])
    print("CLEAN STATS:", out["clean_stats"])
    print()

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CASE: easy_arithmetic
--------------------------------------------------------------------------------
PROMPT PREVIEW:
<|im_start|>system
You are a precise, helpful assistant. Always reason step by step inside <think></think> tags, then write your final answer inside <answer></answer> tags.<|im_end|>
<|im_start|>user
What is 7 + 5? Return output exactly in <think> and <answer> tags.<|im_end|>
<|im_start|>assistant


RAW OUTPUT:
<think>
The user is asking for the result of 7 + 5, which is a simple arithmetic addition problem. I need to calculate the sum of these two numbers and then present the answer in the requested format.

7 + 5 = 12

I'll put this in the <answer> tags as requested.
</think>

<answer>7 + 5 = 12</answer><|im_end|>

CLEAN OUTPUT:
<think>
The user is asking for the result of 7 + 5, which is a simple arithmetic addition problem. I need to calculate the sum of these two numbers and then present the answer in the requested format.

7 + 5 = 12

I'll put this in the <answer

In [5]:
def compliance_label(stats):
    if stats["thinking_count"] > 0:
        return "FAIL_extra_thinking_tag"
    if (
        stats["has_think_open"]
        and stats["has_think_close"]
        and stats["has_answer_open"]
        and stats["has_answer_close"]
    ):
        return "PASS_full_schema"
    if stats["has_think_open"] or stats["has_answer_open"]:
        return "PARTIAL_schema"
    return "FAIL_no_schema"

summary = []
for r in results:
    summary.append({
        "name": r["name"],
        "raw_result": compliance_label(r["raw_stats"]),
        "clean_result": compliance_label(r["clean_stats"]),
        "raw_stats": r["raw_stats"],
        "clean_stats": r["clean_stats"],
    })

print(json.dumps(summary, indent=2))

[
  {
    "name": "easy_arithmetic",
    "raw_result": "PASS_full_schema",
    "clean_result": "PASS_full_schema",
    "raw_stats": {
      "has_think_open": true,
      "has_think_close": true,
      "has_answer_open": true,
      "has_answer_close": true,
      "think_count": 1,
      "thinking_count": 0,
      "answer_count": 2,
      "im_end_count": 1
    },
    "clean_stats": {
      "has_think_open": true,
      "has_think_close": true,
      "has_answer_open": true,
      "has_answer_close": true,
      "think_count": 1,
      "thinking_count": 0,
      "answer_count": 2,
      "im_end_count": 0
    }
  },
  {
    "name": "egg_problem",
    "raw_result": "PARTIAL_schema",
    "clean_result": "PARTIAL_schema",
    "raw_stats": {
      "has_think_open": true,
      "has_think_close": true,
      "has_answer_open": true,
      "has_answer_close": false,
      "think_count": 1,
      "thinking_count": 0,
      "answer_count": 1,
      "im_end_count": 1
    },
    "clean_stats": {
  

In [6]:
def classify_case(raw_stats, text):
    if raw_stats["thinking_count"] > 0:
        return "FAIL_thinking_leak"
    if raw_stats["think_count"] > 1:
        return "FAIL_duplicate_think"
    if (
        raw_stats["has_think_open"]
        and raw_stats["has_think_close"]
        and raw_stats["has_answer_open"]
        and raw_stats["has_answer_close"]
    ):
        return "PASS_exact"
    if raw_stats["has_think_open"] and raw_stats["has_think_close"]:
        return "PARTIAL_only_think"
    if raw_stats["has_answer_open"] or raw_stats["has_answer_close"]:
        return "PARTIAL_only_answer"
    return "FAIL_no_schema"

report = []
for r in results:
    verdict = classify_case(r["raw_stats"], r["decoded_raw"])
    report.append({
        "name": r["name"],
        "verdict": verdict,
        "raw_stats": r["raw_stats"],
    })

for row in report:
    print(row)

{'name': 'easy_arithmetic', 'verdict': 'PASS_exact', 'raw_stats': {'has_think_open': True, 'has_think_close': True, 'has_answer_open': True, 'has_answer_close': True, 'think_count': 1, 'thinking_count': 0, 'answer_count': 2, 'im_end_count': 1}}
{'name': 'egg_problem', 'verdict': 'PARTIAL_only_think', 'raw_stats': {'has_think_open': True, 'has_think_close': True, 'has_answer_open': True, 'has_answer_close': False, 'think_count': 1, 'thinking_count': 0, 'answer_count': 1, 'im_end_count': 1}}
{'name': 'format_only', 'verdict': 'FAIL_thinking_leak', 'raw_stats': {'has_think_open': True, 'has_think_close': True, 'has_answer_open': False, 'has_answer_close': False, 'think_count': 2, 'thinking_count': 2, 'answer_count': 0, 'im_end_count': 0}}
{'name': 'explicit_schema', 'verdict': 'PASS_exact', 'raw_stats': {'has_think_open': True, 'has_think_close': True, 'has_answer_open': True, 'has_answer_close': True, 'think_count': 1, 'thinking_count': 0, 'answer_count': 1, 'im_end_count': 1}}
